<a href="https://colab.research.google.com/github/ysasson-portfolio/text-analytics-spring-2026/blob/main/assignment_5/notebooks/Yarden_Sasson_A5_OptionB_Job_Fit_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 — Option B: Job Fit Analyzer
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Yarden Sasson
**Date:** May 13, 2026  
**Option:** B — Job Fit Analyzer  
**API Path:** Paid

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Load Job Descriptions and Resume](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Analysis Prompts and Chain](#5-analysis)
6. [Zero-shot vs. Few-shot Comparison](#6-comparison)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the analysis pipeline.  
> See the Option B Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `pandas`, `sentence-transformers` (free path)

In [129]:
# ── Install packages (uncomment as needed) ──
!pip install langchain langchain-openai chromadb pypdf python-dotenv sentence-transformers
!pip install -q langchain langchain-community
# ── Load API keys from .env ──
import os
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
from langchain_core.documents import Document
import requests

# ── Your imports below ──

In [130]:
#Clone the Repository (only needs to happen once)
!git clone https://github.com/ysasson-portfolio/text-analytics-spring-2026.git

fatal: destination path 'text-analytics-spring-2026' already exists and is not an empty directory.


In [131]:
#If the github gets updated and I want the latest changes without re-cloning
%cd /content/text-analytics-spring-2026
!git pull

/content/text-analytics-spring-2026
Already up to date.


---
<a id="2-loading"></a>
## 2. Load Job Descriptions and Resume

**Required:**
- 10+ JD files in `data/job_descriptions/` (each as a separate .txt or .pdf)
- Your resume in `data/resume/`
- A metadata file `data/jd_metadata.csv` with columns: filename, company, title, source_url, date_collected

Print: number of JDs loaded, number of resume docs, and preview content from each.

In [132]:
# ── Load JD metadata ──
metadata_df = pd.read_csv("https://raw.githubusercontent.com/ysasson-portfolio/text-analytics-spring-2026/refs/heads/main/assignment_5/data/jd_metadata.csv")

print(metadata_df)

                                           Job Title  \
0                      Business Intelligence Analyst   
1  Business Intelligence Analyst, Sports - Brand ...   
2                      Business Intelligence Analyst   
3                                   Business Analyst   
4                                   Business Analyst   
5                                   Business Analyst   
6                                 Business Analyst I   
7                Sr. Analyst, Strategy and Analytics   
8                                 Strategy Associate   
9                                  Manager, Strategy   

                                    Company  \
0                             Guitar Center   
1                   Creative Artists Agency   
2  Los Angeles Tourism and Convention Board   
3             Red Bull Distribution Company   
4                                   Hadrian   
5                        Polestar Analytics   
6                  Skyworks Solutions, Inc.   
7      

In [133]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

folder_path = "/content/text-analytics-spring-2026/assignment_5/data/job_descriptions"

loader = DirectoryLoader(
    folder_path,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print(len(documents))

10


In [134]:
documents[0]

Document(metadata={'source': '/content/text-analytics-spring-2026/assignment_5/data/job_descriptions/Business Analyst-Polestar Analytics.txt'}, page_content="About the job\nJob Title: Business Analyst - Retail\n\nLocation: Los Angeles\n\nJob Type: Full-time\n\nExperience: 2+ years\n\nIndustry: Analytics Services\n\n\n\nRoles and Responsibilities:\n\nWork closely with clients and internal stakeholders to gather, analyse, and document business requirements.\nTranslate business needs into functional specifications, user stories, and process flows with a strong focus on data-driven decision making.\nSupport consulting engagements by conducting market research, competitor benchmarking, and industry analysis.\nCollaborate with Product, Data Engineering, and Analytics teams to define solution approaches aligned with client objectives.\nAssist in creating business cases, value propositions, and solution decks for client presentations.\nParticipate in workshops, stakeholder discussions, and req

In [135]:
import re

job_rows = []

for doc in documents:

    file_name = os.path.basename(doc.metadata["source"])
    file_name = file_name.replace(".txt", "")

    # Clean text
    cleaned_text = doc.page_content

    # Replace newlines/tabs with spaces
    cleaned_text = cleaned_text.replace("\n", " ")
    cleaned_text = cleaned_text.replace("\t", " ")

    # Remove extra whitespace
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    job_rows.append({
        "File Name": file_name,
        "job_description": cleaned_text
    })

job_df = pd.DataFrame(job_rows)

job_df.head()

,File Name,job_description
0,Business Analyst-Polestar Analytics,About the job Job Title: Business Analyst - Re...
1,Strategy Manager- Paramount,"The Manager, Strategy will support the develop..."
2,Business Analyst-RedBull,"In this role, the successful candidate will be..."
3,"Sr. Analyst, Strategy and Analytics-Sofi Stadium","About Hollywood Park Hollywood Park, a near 30..."
4,Strategy Associate-Cedars Sinai,Align yourself with an organization that has a...


In [136]:
merged_df = job_df.merge(
    metadata_df,
    on="File Name",
    how="left"
)

merged_df

,File Name,job_description,Job Title,Company,Source URL,Date Collected,Role Type
0,Business Analyst-Polestar Analytics,About the job Job Title: Business Analyst - Re...,Business Analyst,Polestar Analytics,https://www.linkedin.com/jobs/view/4399610949/,4/28/2026,Business Analyst
1,Strategy Manager- Paramount,"The Manager, Strategy will support the develop...","Manager, Strategy",Paramount,https://www.linkedin.com/jobs/view/4398583482/,4/28/2026,Strategy Analyst
2,Business Analyst-RedBull,"In this role, the successful candidate will be...",Business Analyst,Red Bull Distribution Company,https://www.linkedin.com/jobs/view/4405852876/,4/28/2026,Business Analyst
3,"Sr. Analyst, Strategy and Analytics-Sofi Stadium","About Hollywood Park Hollywood Park, a near 30...","Sr. Analyst, Strategy and Analytics",SoFi Stadium and Hollywood Park,https://www.linkedin.com/jobs/view/4388281012/,4/28/2026,Strategy Analyst
4,Strategy Associate-Cedars Sinai,Align yourself with an organization that has a...,Strategy Associate,Cedars Sinai,https://www.linkedin.com/jobs/view/4400809577/,4/28/2026,Strategy Analyst
5,Business Intelligence Analyst-CAA,Who We Are Creative Artists Agency (CAA) is th...,"Business Intelligence Analyst, Sports - Brand ...",Creative Artists Agency,https://www.linkedin.com/jobs/view/4402204259/,4/28/2026,Business Intelligence Analyst
6,Business Analyst-Skyworks,If you are looking for a challenging and excit...,Business Analyst I,"Skyworks Solutions, Inc.",https://www.linkedin.com/jobs/view/4403514344/,4/28/2026,Business Analyst
7,Business Analyst-Hadrian,Hadrian is building autonomous factories that ...,Business Analyst,Hadrian,https://www.linkedin.com/jobs/view/4402613474/,4/28/2026,Business Analyst
8,Business Inteligence Analyst-Guitar Center,"About the Role: At Guitar Center, the Data Tea...",Business Intelligence Analyst,Guitar Center,https://www.linkedin.com/jobs/collections/reco...,4/28/2026,Business Intelligence Analyst
9,Business Intelligence Analyst- LA Tourism and ...,WHO WE ARE The mission of the Los Angeles Tour...,Business Intelligence Analyst,Los Angeles Tourism and Convention Board,https://www.linkedin.com/jobs/view/4366060859/,4/28/2026,Business Intelligence Analyst


In [137]:
merged_df["job_description"]

,job_description
0,About the job Job Title: Business Analyst - Re...
1,"The Manager, Strategy will support the develop..."
2,"In this role, the successful candidate will be..."
3,"About Hollywood Park Hollywood Park, a near 30..."
4,Align yourself with an organization that has a...
5,Who We Are Creative Artists Agency (CAA) is th...
6,If you are looking for a challenging and excit...
7,Hadrian is building autonomous factories that ...
8,"About the Role: At Guitar Center, the Data Tea..."
9,WHO WE ARE The mission of the Los Angeles Tour...


In [138]:
folder_path_resume = "/content/text-analytics-spring-2026/assignment_5/data/resume"

loader_resume = DirectoryLoader(
    folder_path_resume,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

resume_data = loader_resume.load()

print(len(resume_data))

1


In [139]:
import re


# Clean text
cleaned_text = resume_data[0].page_content

# Replace newlines/tabs with spaces
cleaned_text = cleaned_text.replace("\n", " ")
cleaned_text = cleaned_text.replace("\t", " ")

# Remove extra whitespace
resume = re.sub(r"\s+", " ", cleaned_text).strip()


In [140]:
resume

'Yarden Sasson EDUCATION Loyola Marymount University Masters of Science in Business Analytics Class of 2026 • Relevant Coursework: Data Management for Business Intelligence, Introduction to Machine Learning, Strategic Integration Analytics • Honors Society: Beta Gamma Sigma University of California, Los Angeles (UCLA) Bachelors of Science in Cognitive Science with a Specialization in Computing Class of 2020 • Relevant Coursework: Advanced Topics in MATLAB Programming for Behavioral Sciences, Science of Language, and Neural Networks • Activities: Den Operations Club, UCLA’s The Den, UCLA Hillel WORK EXPERIENCE Israel Economic and Trade Mission to the West Coast Head of Innovation, Los Angeles (May 2023-Present) • Conduct and analyze market research to provide actionable insights into business strategies • Organize high-impact business events with multiple partners and B2B meetings including national pavilions at CES, NRF, RSA, HLTH, and CyberWeek TLV Conferences. • Facilitate go-to-mark

In [141]:
# ── Preview sample content ──


---
<a id="3-chunking"></a>
## 3. Text Chunking

Split your documents into chunks.  
**Required:** Try at least 2 chunking strategies, compare them quantitatively, and justify your final choice.

**Hint:** JDs often have natural sections (Requirements, Responsibilities, Qualifications). Consider whether your splitter respects these boundaries.

In [143]:
# ── Strategy 1 ──
def chunk_text(text, chunk_size=300, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk.strip())
        start += chunk_size - overlap
    return [c for c in chunks if len(c) > 20]  # skip tiny fragments

fixed_chunks = []

for doc in documents:
    chunks = chunk_text(doc.page_content, chunk_size=800, overlap=100)

    for i, chunk in enumerate(chunks):
        fixed_chunks.append({
            "text": chunk,
            "filename": doc.metadata.get("filename", ""),
            "company": doc.metadata.get("company", ""),
            "title": doc.metadata.get("title", ""),
            "doc_type": doc.metadata.get("doc_type", ""),
            "chunk_id": i,
            "chunk_strategy": "fixed_size"
        })

print("Strategy 1 chunks:", len(fixed_chunks))
print(fixed_chunks[0]["text"][:300])



Strategy 1 chunks: 66
About the job
Job Title: Business Analyst - Retail

Location: Los Angeles

Job Type: Full-time

Experience: 2+ years

Industry: Analytics Services



Roles and Responsibilities:

Work closely with clients and internal stakeholders to gather, analyse, and document business requirements.
Translate bus


In [158]:
# ── Strategy 2 ──
def chunk_text_by_sentences(text, chunk_size=300, overlap_words=20):
    """Split text into chunks that respect sentence boundaries."""
    import re
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        if len(current_chunk) + len(sentence) > chunk_size and current_chunk:
            chunks.append(current_chunk.strip())
            # Keep overlap by taking the end of the current chunk
            words = current_chunk.split()
            overlap_text = " ".join(words[-overlap_words:]) if len(words) > overlap_words else current_chunk
            current_chunk = overlap_text + " " + sentence
        else:
            current_chunk += (" " if current_chunk else "") + sentence

    if current_chunk.strip():
        chunks.append(current_chunk.strip())

    return chunks

sentence_chunks = []

for doc in documents:
    chunks = chunk_text_by_sentences(doc.page_content, chunk_size=800, overlap_words=20)

    for i, chunk in enumerate(chunks):
        sentence_chunks.append({
            "text": chunk,
            "filename": doc.metadata.get("filename", ""),
            "company": doc.metadata.get("company", ""),
            "title": doc.metadata.get("title", ""),
            "doc_type": doc.metadata.get("doc_type", ""),
            "chunk_id": i,
            "chunk_strategy": "sentence_aware"
        })

print("Strategy 2 chunks:", len(sentence_chunks))
print(sentence_chunks[0]["text"][:300])

Strategy 2 chunks: 70
About the job
Job Title: Business Analyst - Retail

Location: Los Angeles

Job Type: Full-time

Experience: 2+ years

Industry: Analytics Services



Roles and Responsibilities:

Work closely with clients and internal stakeholders to gather, analyse, and document business requirements. Translate bus


In [150]:
# ── Compare strategies ──


In [160]:
test_doc = documents[0].page_content

chunks_fixed = chunk_text(test_doc, chunk_size=800, overlap=50)
chunks_sentence = chunk_text_by_sentences(test_doc, chunk_size=800, overlap_words=20)

print(f"Fixed-size:  {len(chunks_fixed)} chunks")
print(f"Sentence:    {len(chunks_sentence)} chunks")

print("\n" + "=" * 60)
print("FIXED-SIZE chunks:")
print("=" * 60)
for i, c in enumerate(chunks_fixed):
    print(f"\nChunk {i+1} ({len(c)} chars): {c[:100]}...")

print("\n" + "=" * 60)
print("SENTENCE-AWARE chunks:")
print("=" * 60)
for i, c in enumerate(chunks_sentence):
    print(f"\nChunk {i+1} ({len(c)} chars): {c[:100]}...")


# ── Chunk ALL documents using sentence-aware strategy ──
# Each chunk keeps track of which document it came from (metadata)

all_chunks = []  # list of dicts: {"text": ..., "source": ...}

for doc in documents:
    doc_chunks = chunk_text_by_sentences(doc.page_content, chunk_size=400, overlap_words=20)
    for chunk in doc_chunks:
        all_chunks.append({
            "text": chunk,
            "source": doc.metadata['source']
        })

print(f"Total chunks across all documents: {len(all_chunks)}")
print(f"\nChunks per source:")
from collections import Counter
for source, count in Counter(c["source"] for c in all_chunks).items():
    print(f"  {source}: {count}")


Fixed-size:  6 chunks
Sentence:    8 chunks

FIXED-SIZE chunks:

Chunk 1 (800 chars): About the job
Job Title: Business Analyst - Retail

Location: Los Angeles

Job Type: Full-time

Expe...

Chunk 2 (800 chars): resentations.
Participate in workshops, stakeholder discussions, and requirement elicitation session...

Chunk 3 (800 chars): ising, supply chain, demand planning, customer analytics) 
Exposure to Product Management concepts s...

Chunk 4 (800 chars): prise planning powerhouse, Polestar Solutions helps its customers bring out the most sophisticated i...

Chunk 5 (800 chars): des.



The list includes:

Recognized as “Great Place to Work” - 2024
Recognized as the Top 50 Comp...

Chunk 6 (129 chars): poster

• Commute to this job’s location

• Can start immediately

• Working in an onsite setting

•...

SENTENCE-AWARE chunks:

Chunk 1 (763 chars): About the job
Job Title: Business Analyst - Retail

Location: Los Angeles

Job Type: Full-time

Expe...

Chunk 2 (672 chars): define s

In [154]:
documents

[Document(metadata={'source': '/content/text-analytics-spring-2026/assignment_5/data/job_descriptions/Business Analyst-Polestar Analytics.txt'}, page_content="About the job\nJob Title: Business Analyst - Retail\n\nLocation: Los Angeles\n\nJob Type: Full-time\n\nExperience: 2+ years\n\nIndustry: Analytics Services\n\n\n\nRoles and Responsibilities:\n\nWork closely with clients and internal stakeholders to gather, analyse, and document business requirements.\nTranslate business needs into functional specifications, user stories, and process flows with a strong focus on data-driven decision making.\nSupport consulting engagements by conducting market research, competitor benchmarking, and industry analysis.\nCollaborate with Product, Data Engineering, and Analytics teams to define solution approaches aligned with client objectives.\nAssist in creating business cases, value propositions, and solution decks for client presentations.\nParticipate in workshops, stakeholder discussions, and re

### Chunking Decision

**Which strategy did you choose?**  
**Why?**  
**Final settings (chunk_size, overlap):**

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

Embed your chunks and store them in a vector database (ChromaDB or FAISS).

**Paid path:** OpenAI `text-embedding-3-small`  
**Free path:** `sentence-transformers/all-MiniLM-L6-v2`

After creating the store, run a test similarity search to verify it works.

In [ ]:
# ── Create embeddings and vector store ──


In [ ]:
# ── Verify: run a test similarity search ──


---
<a id="5-analysis"></a>
## 5. Analysis Prompts and Chain

Build 3 analysis types, each with its own prompt (one iteration for 3 types) or 3 iterations for 1 type with its own prompt:

1. **Skill Gap Report:** Compare resume skills vs. JD requirements. Output matching skills, missing skills, and recommended actions.
2. **Keyword Alignment:** Extract key terms from a JD, check which appear in the resume, report a match rate.
3. **Fit Summary:** 3-4 sentence narrative assessment citing evidence from both documents.

You also need to wire up the LLM and a way to pass a specific JD + resume into each prompt.

**Required:** Document at least 3 prompt iterations total (across any analysis type) with rationale.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [ ]:
# ── Initialize LLM ──
from sentence_transformers import SentenceTransformer

In [ ]:
# ── Analysis 1: Skill Gap Report ──
#Give me a analysis of the skill gap between the resume inputted and the job description (Baseline)
#Compare and Contrast the resume to the job posting and let me know where the skill gap needs to improve (Iteration 1)
#Explain what is missing from my resume and why it is important for this position (Iteration 2)
#If I have the skills for this position, do I have enough experience for the position as well? (Iteration 3) (Potentially add more and be aware of the changes)

In [ ]:
# ── Analysis 2: Keyword Alignment ──


In [ ]:
# ── Analysis 3: Fit Summary ──


### Prompt Iteration Log

Document at least 3 total iterations across any of the analysis types.

**Iteration 1:** [Which analysis? What changed? Why? What improved?]

**Iteration 2:** [Which analysis? What changed? Why? What improved?]

**Iteration 3:** [Which analysis? What changed? Why? What improved?]

---
<a id="6-comparison"></a>
## 6. Zero-shot vs. Few-shot Comparison

Pick one of your 3 analysis types. Create a few-shot version by adding 1-2 example input/output pairs to the prompt. Run both versions on the same JD and compare outputs.

**Reminder:** You must write the few-shot examples yourself (Tier 2).

In [ ]:
# ── Few-shot version of your chosen analysis ──
# Based on the following job description, identify whether the job is match from the resume.

#Job Description: (insert job description here)
#Match: Yes because ....

#Job Description: (insert job description here)
#Match: No because ....

In [ ]:
# ── Run both on the same JD, display side by side ──


### Zero-shot vs. Few-shot Analysis

**Which analysis type did you compare?**

**Which performed better?**

**Why? (use specific examples from the outputs above)**

---
<a id="7-evaluation"></a>
## 7. Evaluation

Run all 3 analysis types on your **top 3 target JDs** (9 total analyses).

For each, score:
- **Retrieval relevance:** Did it pull the right JD sections? (Yes/Partial/No)
- **Skill identification accuracy:** Are identified skills/gaps correct? (count correct vs. incorrect)
- **Actionability:** Are recommendations specific and useful? (1-5)
- **Faithfulness:** Does output stick to document content? (Faithful/Partial/Hallucinated)

**Reminder:** Evaluation must be your own work (Tier 2 — AI prohibited).

In [ ]:
# ── Run 9 analyses (3 JDs x 3 analysis types) ──


In [ ]:
# ── Summarize evaluation results ──


### Evaluation Analysis

**Which analysis type worked best?**

**Which JDs produced the best/worst results? Why?**

**Where did the system hallucinate or produce inaccurate results?**

**What would you improve?**

---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option B*